# 03 — Modelos base

Este notebook documenta M1–M4 paso a paso: preparación, evaluación temporal, referencias simples y TF-IDF.

Un **experimento** es una ejecución con datos, variables y parámetros definidos. Su registro permite saber exactamente cómo se obtuvo un resultado.

## 1. Configuración única

La semilla y las rutas se guardan en `configs/modeling.yaml`. Una **semilla** es un número que permite repetir los mismos pasos aleatorios.

In [ ]:
from src.evaluation.experiment import load_experiment_config, set_seed

config = load_experiment_config()
config

## 2. Comprobar la semilla

Al reiniciar la misma semilla, Python y NumPy deben producir los mismos números. Esto no garantiza que todo modelo sea idéntico, pero elimina una fuente común de variación.

In [ ]:
import random
import numpy as np

seed = config['experiment']['seed']
set_seed(seed)
primera_prueba = (random.random(), np.random.random())

set_seed(seed)
segunda_prueba = (random.random(), np.random.random())

assert primera_prueba == segunda_prueba
primera_prueba

## 3. Información mínima de cada ejecución

Cada ejecución registrará en MLflow:

- versión del código en Git;
- versión de los datos en DVC;
- objetivo y periodo evaluado;
- vista completa o sin texto compartido;
- variables de entrada;
- semilla;
- ejecución local o en Khipu;
- parámetros y métricas del modelo.

Una **métrica** es un número usado para evaluar un resultado. Por ejemplo, Macro-F1 para T1.

In [ ]:
from src.evaluation.experiment import build_run_record

registro_ejemplo = build_run_record(
    config=config,
    run_name='ejemplo-no-publicar',
    stage='M1',
    target='experiment_setup',
    split='not_applicable',
    view='not_applicable',
    features=[],
)
registro_ejemplo

## 4. Ejecuciones en Khipu

Los nodos SLURM no tienen acceso a internet. El trabajo guarda primero un registro JSON local. Al terminar, ese registro se publica en MLflow desde el nodo de acceso.

Esto separa dos acciones:

1. **Ejecutar:** calcular el resultado dentro de SLURM.
2. **Publicar:** enviar parámetros, métricas y artefactos pequeños a MLflow.

Los artefactos grandes, como embeddings o índices FAISS, se guardarán con DVC.

# M2 — Evaluación temporal

Una **división temporal** separa los datos por fecha. El modelo aprende con el pasado y se evalúa con datos posteriores. Esto representa mejor el uso real que una división aleatoria.

## 5. Periodos congelados

- **Ajuste:** enero de 2023 a septiembre de 2024. El modelo aprende aquí.
- **Calibración:** octubre a diciembre de 2024. Aquí se eligen parámetros y umbrales.
- **Validación temporal:** enero a junio de 2025. Mide el comportamiento posterior.

Un **umbral** convierte una probabilidad en una decisión. Por ejemplo, un umbral de 0.30 marca como positivo un caso con probabilidad de 0.35.

In [ ]:
import json
from src.evaluation.experiment import PROJECT_ROOT

ruta_contrato = PROJECT_ROOT / config['paths']['evaluation_contract']
evaluacion = json.loads(ruta_contrato.read_text(encoding='utf-8'))
evaluacion['splits']

## 6. Dos vistas de evaluación

La vista **completa** conserva todos los casos elegibles. La vista **sin texto compartido** excluye narrativas que ya aparecieron en periodos usados para aprender.

La segunda será la vista principal para elegir modelos porque reduce el beneficio artificial de memorizar plantillas. La vista completa también se conserva porque las plantillas existen en la operación real.

In [ ]:
import pandas as pd

pd.DataFrame(evaluacion['row_counts']).T

## 7. Métricas congeladas

- **Macro-F1 (T1):** calcula F1 por motivo y da el mismo peso a cada motivo.
- **Top-3 (T1):** revisa si el motivo correcto aparece entre tres sugerencias.
- **Average precision o precisión promedio (T2–T4):** resume la relación entre precisión y cobertura.
- **Precisión:** de los casos marcados positivos, cuántos eran positivos.
- **Cobertura o recall:** de los positivos reales, cuántos encontró el modelo.
- **Brier score:** mide el error de las probabilidades; un valor menor es mejor.

No usaremos el porcentaje total de aciertos como métrica principal porque T3 y T4 tienen pocos positivos. Un modelo que siempre diga 'no' tendría un porcentaje alto, pero no ayudaría al triaje.

In [ ]:
resumen_objetivos = {}
for periodo in ('fit', 'calibration', 'validation'):
    resumen_objetivos[periodo] = {}
    for objetivo, valores in evaluacion['targets'][periodo]['no_shared_text'].items():
        resumen_objetivos[periodo][objetivo] = {
            clave: valor
            for clave, valor in valores.items()
            if clave != 'class_counts'
        }
resumen_objetivos

## 8. Variables permitidas

La primera comparación usará narrativa normalizada y producto canónico. Empresa, estado y fecha serán candidatos: solo se añadirán si demuestran valor.

Se excluyen `Issue`, respuestas de la empresa y resultados T1–T4. Esas columnas revelarían la respuesta que el modelo intenta predecir; usar esa información se llama **fuga de información**.

In [ ]:
evaluacion['features']

# M3 — Referencias simples

Una **referencia** es un método sencillo que fija el resultado mínimo que debe superar un modelo más complejo.

Comparamos dos referencias que no leen la narrativa:

- **Frecuencia global:** siempre usa el resultado más común del ajuste.
- **Frecuencia por producto:** usa el resultado histórico más común dentro de cada producto.

La regla por producto se parece a una regla sencilla de call center, pero no representa las reglas privadas de un banco. Solo resume las frecuencias del archivo CFPB.

In [ ]:
ruta_referencias = PROJECT_ROOT / config['paths']['baseline_report']
referencias = json.loads(ruta_referencias.read_text(encoding='utf-8'))

In [ ]:
filas = []
for objetivo, modelos in referencias['results'].items():
    metrica = 'macro_f1' if objetivo == 'T1' else 'average_precision'
    for modelo, resultado in modelos.items():
        valor = resultado['metrics']['validation']['no_shared_text'][metrica]
        filas.append({
            'objetivo': objetivo,
            'referencia': modelo,
            'métrica': metrica,
            'valor': valor,
        })
pd.DataFrame(filas)

## 9. Qué muestra T1

La regla por producto encuentra el motivo correcto entre sus tres opciones en 90.98% de la validación sin texto compartido. Sin embargo, su Macro-F1 al elegir un único motivo es solo 0.0684.

Interpretación: el producto reduce mucho las opciones, pero no basta para ordenar correctamente los motivos dentro de cada producto. La narrativa debe demostrar valor en esa decisión más precisa.

In [ ]:
detalle_producto = []
for objetivo, modelos in referencias['results'].items():
    metricas = modelos['product_frequency']['metrics']['validation']['no_shared_text']
    detalle_producto.append({'objetivo': objetivo, **metricas})
pd.DataFrame(detalle_producto)

## 10. Qué muestran T2–T4

El producto también contiene señal para los resultados binarios. La precisión promedio sube frente a la frecuencia global:

- T2: de 0.3557 a 0.4509.
- T3: de 0.0133 a 0.1272.
- T4: de 0.0108 a 0.1208.

Aun así, no es una decisión final. En T3 la regla logra 11.82% de precisión con 80.49% de cobertura. En T4 logra 18.35% de precisión con 41.38% de cobertura. El modelo de texto deberá mejorar este equilibrio.

# M4 — TF-IDF

**TF-IDF** representa una narrativa mediante la importancia de sus palabras y pares de palabras. M4 combina esa representación con el producto.

El clasificador es **lineal**: suma evidencia a favor o en contra de cada resultado. Se entrenó en CPU porque estas matrices contienen muchos ceros y no aprovechan la A100 como lo hará BGE.

In [ ]:
ruta_tfidf = PROJECT_ROOT / config['paths']['tfidf_report']
tfidf = json.loads(ruta_tfidf.read_text(encoding='utf-8'))

In [ ]:
comparacion = []
for objetivo, resultado in tfidf['results'].items():
    metrica = 'macro_f1' if objetivo == 'T1' else 'average_precision'
    comparacion.append({
        'objetivo': objetivo,
        'métrica': metrica,
        'regla por producto': resultado['product_baseline_primary_metric'],
        'TF-IDF + producto': resultado['metrics']['validation']['no_shared_text'][metrica],
        'cambio': resultado['primary_metric_improvement'],
    })
pd.DataFrame(comparacion)

## 11. La narrativa aporta valor para T1–T3

- T1: Macro-F1 sube de 0.0684 a 0.1984 y top-3 de 90.98% a 95.92%.
- T2: precisión promedio sube de 0.4509 a 0.5744.
- T3: precisión promedio sube de 0.1272 a 0.2904.

T1 todavía tiene Macro-F1 bajo por la cantidad de motivos y sus tamaños muy diferentes. Sin embargo, el top-3 mejora y puede ayudar al agente mostrando una lista corta en vez de decidir automáticamente.

In [ ]:
detalle_tfidf = []
for objetivo, resultado in tfidf['results'].items():
    metricas = resultado['metrics']['validation']['no_shared_text']
    detalle_tfidf.append({'objetivo': objetivo, **metricas})
pd.DataFrame(detalle_tfidf)

## 12. TF-IDF no mejora T4

T4 baja de 0.1208 con la regla por producto a 0.0781 con TF-IDF. El texto había mejorado en calibración, pero no mantuvo esa ventaja en 2025-H1.

Esto indica inestabilidad temporal. No sabemos la causa solo con estos datos, así que la decisión correcta es no promover esta versión para T4 y conservar la regla por producto como referencia.

## 13. Artefacto reproducible

El vocabulario, el codificador de producto y los cuatro modelos ocupan aproximadamente 101 MB. Se guardan con DVC en `artifacts/models/tfidf`, no dentro de Git.

Hash DVC: `94536b940d1546580c62b80078e2bbfc.dir`.

## Siguiente paso

M5 comparará BGE contra TF-IDF sobre las mismas filas. BGE solo se conservará donde demuestre una mejora suficiente para justificar su mayor costo.